# 🎯 Lead Scoring Model Training
## Train YOUR OWN AI Model for Sales Lead Prediction

This notebook walks you through:
1. Loading training data
2. Exploring the data
3. Training the ML model
4. Evaluating performance
5. Saving the model for production use

In [ ]:
# Import necessary libraries
import sys
sys.path.append('../backend')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from app.ml.lead_scorer import LeadScorer
from app.ml.generate_sample_data import generate_sample_leads

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Imports successful!")

## Step 1: Generate/Load Training Data

In [ ]:
# Generate sample leads (or load your own data)
leads_df = generate_sample_leads(n_samples=500)

print(f"\n📊 Dataset Shape: {leads_df.shape}")
print(f"Columns: {leads_df.columns.tolist()}")

# Display first few rows
leads_df.head()

## Step 2: Explore the Data

In [ ]:
# Conversion statistics
print("\n📈 Conversion Statistics:")
print(f"Total Leads: {len(leads_df)}")
print(f"Converted: {leads_df['converted'].sum()} ({leads_df['converted'].mean()*100:.1f}%)")
print(f"Not Converted: {(~leads_df['converted']).sum()} ({(~leads_df['converted']).mean()*100:.1f}%)")

# Plot conversion rate
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Conversion by source
source_conv = leads_df.groupby('source')['converted'].mean().sort_values(ascending=False)
source_conv.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Conversion Rate by Source')
axes[0].set_ylabel('Conversion Rate')
axes[0].set_xlabel('Source')

# Conversion by industry
industry_conv = leads_df.groupby('industry')['converted'].mean().sort_values(ascending=False)
industry_conv.plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Conversion Rate by Industry')
axes[1].set_ylabel('Conversion Rate')
axes[1].set_xlabel('Industry')

# Conversion by company size
size_conv = leads_df.groupby('company_size')['converted'].mean().sort_values(ascending=False)
size_conv.plot(kind='bar', ax=axes[2], color='lightgreen')
axes[2].set_title('Conversion Rate by Company Size')
axes[2].set_ylabel('Conversion Rate')
axes[2].set_xlabel('Company Size')

plt.tight_layout()
plt.show()

In [ ]:
# Engagement metrics analysis
engagement_cols = ['website_visits', 'pages_viewed', 'email_opens', 'email_clicks', 'form_submissions']

fig, axes = plt.subplots(1, len(engagement_cols), figsize=(20, 4))

for idx, col in enumerate(engagement_cols):
    converted = leads_df[leads_df['converted']][col]
    not_converted = leads_df[~leads_df['converted']][col]
    
    axes[idx].hist([not_converted, converted], label=['Not Converted', 'Converted'], bins=15, alpha=0.7)
    axes[idx].set_title(col.replace('_', ' ').title())
    axes[idx].legend()
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Count')

plt.tight_layout()
plt.show()

print("\n💡 Notice: Converted leads typically have higher engagement!")

## Step 3: Train the ML Model 🚀

In [ ]:
# Initialize the lead scorer
scorer = LeadScorer(model_type='xgboost')  # Try: 'xgboost', 'random_forest', 'gradient_boosting'

# Train the model
metrics = scorer.train(leads_df, test_size=0.2)

print("\n✅ Training Complete!")
print(f"\nModel Metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value}")

## Step 4: Analyze Feature Importance

In [ ]:
# Plot feature importance
if scorer.feature_importance is not None:
    plt.figure(figsize=(10, 8))
    feature_imp = scorer.feature_importance.head(10)
    plt.barh(feature_imp['feature'], feature_imp['importance'], color='skyblue')
    plt.xlabel('Importance')
    plt.title('Top 10 Most Important Features for Lead Scoring')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print("\n🎯 These features have the most impact on lead conversion predictions!")

## Step 5: Test the Model on Sample Leads

In [ ]:
# Create test leads with different profiles
test_leads = [
    {
        'first_name': 'Ahmed',
        'last_name': 'CEO',
        'email': 'ahmed@techcorp.com',
        'company_name': 'TechCorp',
        'company_size': '201-500',
        'industry': 'technology',
        'job_title': 'CEO',
        'source': 'referral',
        'website_visits': 8,
        'pages_viewed': 15,
        'email_opens': 5,
        'email_clicks': 3,
        'form_submissions': 2,
        'estimated_budget': 100000,
        'timeline': 'immediate',
        'linkedin_profile': 'linkedin.com/in/ahmed',
        'created_at': pd.Timestamp.now()
    },
    {
        'first_name': 'Sarah',
        'last_name': 'Coordinator',
        'email': 'sarah@gmail.com',
        'company_name': 'SmallCo',
        'company_size': '1-10',
        'industry': 'retail',
        'job_title': 'Coordinator',
        'source': 'cold_outreach',
        'website_visits': 1,
        'pages_viewed': 2,
        'email_opens': 0,
        'email_clicks': 0,
        'form_submissions': 0,
        'estimated_budget': None,
        'timeline': '6+ months',
        'linkedin_profile': None,
        'created_at': pd.Timestamp.now()
    }
]

print("\n🎯 Testing Model on Sample Leads:\n")
for lead in test_leads:
    score = scorer.predict(lead)
    insights = scorer.get_lead_insights(lead)
    
    print(f"Lead: {lead['first_name']} {lead['last_name']} ({lead['job_title']})")
    print(f"  Score: {score*100:.1f}%")
    print(f"  Quality: {'🔥 HIGH' if score >= 0.7 else '⚠️ MEDIUM' if score >= 0.4 else '❄️ LOW'}")
    print(f"  Engagement: {insights['features']['engagement_score']:.2f}")
    print()

## Step 6: Save the Trained Model

In [ ]:
# Save model for production use
scorer.save_model('../backend/app/ml/models/lead_scorer.pkl')

print("\n✅ Model saved successfully!")
print("   You can now use this model in your FastAPI application.")

# Test loading
print("\n🔄 Testing model loading...")
test_scorer = LeadScorer()
test_scorer.load_model('../backend/app/ml/models/lead_scorer.pkl')
print("✅ Model loaded successfully!")

## 🎉 Congratulations!

You've successfully trained YOUR OWN lead scoring AI model!

### Next Steps:
1. ✅ Model is ready to use in the FastAPI backend
2. 🔄 Retrain regularly with new data to improve accuracy
3. 📊 Monitor performance in production
4. 🎯 Adjust features based on your specific sales process

### Key Metrics to Watch:
- **ROC AUC**: How well the model separates converted vs non-converted leads
- **Precision**: Of leads marked as "high quality", how many actually convert
- **Recall**: Of all converted leads, how many did we identify as high quality

### Model Improvements:
- Add more historical data for better predictions
- Include domain-specific features (product interest, demo requests, etc.)
- Experiment with different model types
- Tune hyperparameters for your specific data